Note: Most of this code is implemented by Claude, all I did was log normalization while trying to figure out what other fixes can we make (all these fixes were mentioned in the google forms for the question what fixes i'd make)
Additionally, I'd only quoted my fixes in that so yea.....
wanted to make this fast idk why.


In [61]:
!git clone https://github.com/quest-lab-iisc/CNDiff
%cd CNDiff
!ls


Cloning into 'CNDiff'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 56 (delta 6), reused 51 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 1.06 MiB | 4.07 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/CNDiff/CNDiff/CNDiff
cfg	 models     requirements.txt  saved_models  train
dataset  Readme.md  results.txt       scripts	    utils


In [62]:
!cat Readme.md

# ⏳✨ Conditional Diffusion Model with Nonlinear Data Transformation for Time Series Forecasting 📈🌊

[![ICML 2025](https://img.shields.io/badge/ICML-2025-blue.svg)](https://icml.cc/)

Welcome to the official implementation of our **ICML 2025** paper:  
> **Conditional Diffusion Model with Nonlinear Data Transformation for Time Series Forecasting**  

🎯 Our method blends **Generative Model framework** 🌀 with **non-linear data transformations** 🔄 to unlock **state-of-the-art** forecasting performance across diverse time series datasets.  
Whether it’s climate 🌦, finance 💹, or energy ⚡ — this repo has you covered.

---

## 📜 Table of Contents
- [🚀 OpenReview](https://openreview.net/forum?id=kcUNMKqrCg)
- [📚 Paper](https://openreview.net/attachment?id=kcUNMKqrCg&name=pdf)
- [📂 Dataset](https://drive.google.com/drive/folders/1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy)
- 🛠 Installation
- 💻 Usage
- 🤝 Citation
- 📬 Contact

---

## 📚 Paper
📄 **ICML 2025** — *Conditional Diffusion Model with Nonlinear Dat

In [63]:
!pip install -r requirements.txt

In [64]:
!ls cfg/

electricity.yaml  etth1.yaml  ettm1.yaml  exchange.yaml  weather.yaml


In [65]:
!cat cfg/exchange.yaml

default : &DEFAULT

    seed: 2024 
    run_name: exchange
    test: False
    cfg: None
    data:
        dataset: "exchange"
        split_ratio: [0.7, 0.1, 0.2]    # train, val, test
        feature_dim: 8
        seq_len: 96 
        pred_len: 14 
        data_path: 'data/exchange_rate.csv'
        scale: True    # normalise data
        target: 'OT'   # target column
        features: 'M'
        timeenc: 1
        freq: 'd'
        batch_size: 64 
        test_batch_size: 32
        shuffle: True
        drop_last: True

    train:
        device: "cuda:0"
        optimizer: Adam
        epochs: 100
        lr: 0.001
        patience: 10

    model:
        hidden_dim: 64
        n_emb: 2
        n_heads: 8
        attn_dropout: 0.1
        mlp_ratio: 1
        n_depth: 1

    use_cond: True
        
    diff:
        noise_type: "t_phi"
        beta_schedule: quad
        beta_start: 0.0001
        beta_end: 0.1
        timesteps: 100
        model_type: "CN_Diff" 
        n_cop

In [66]:
import os
os.makedirs('data', exist_ok=True)

In [67]:
import os
print(os.getcwd())

/content/CNDiff/CNDiff/CNDiff


In [68]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [69]:
import os
print(os.getcwd())

/content/CNDiff/CNDiff/CNDiff


In [70]:
!ls /content/drive/MyDrive/all_six_datasets/

electricity  ETT-small	exchange_rate  illness	traffic  weather


In [71]:
%cd /content/CNDiff/
!python3 -m scripts.run_cndiff --cfg ./cfg/exchange.yaml

/content/CNDiff
train 5202
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
val 745
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test 

In [72]:
!ls "/content/drive/MyDrive/all_six_datasets/"

electricity  ETT-small	exchange_rate  illness	traffic  weather


In [73]:
!ls "/content/drive/MyDrive/all_six_datasets/exchange_rate/"

exchange_rate.csv


In [74]:
!cp "/content/drive/MyDrive/all_six_datasets/exchange_rate/exchange_rate.csv" /content/CNDiff/data/

In [75]:
%cd /content/CNDiff
!python3 -m scripts.run_cndiff --cfg ./cfg/exchange.yaml

/content/CNDiff
train 5202
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
val 745
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test 

In [76]:
!sed -n '1,35p' /content/CNDiff/scripts/run_cndiff.py

import argparse
import random
import torch
import torch.nn as nn
import numpy as np
from utils.build_model import build_model
from utils.diffusion_utils import calculate_mse_mae

from configmypy import ConfigPipeline, YamlConfig, ArgparseConfig
from dataset.preprocessing import load_data
from train.train_non_gaussian import train

# Parse the config file path from terminal
parser = argparse.ArgumentParser()
parser.add_argument("--cfg", nargs="?", default=None, help="Path to YAML config file")
args, unknown = parser.parse_known_args()
yaml_cfg_path = args.cfg

# Read the configuration file
pipe = ConfigPipeline(
    [
        YamlConfig(yaml_cfg_path, config_name='default'),
        ArgparseConfig(infer_types=True, config_name=None, config_file=None),
        YamlConfig(config_folder='cfg/')
    ]
)

config = pipe.read_conf()

# seed
random.seed(config.seed)
torch.manual_seed(config.seed)
np.random.seed(config.seed)

torch.backends.cudnn.deterministic = True


In [77]:
with open('/content/CNDiff/scripts/run_cndiff.py', 'r') as f:
    content = f.read()

content = content.replace(
    "YamlConfig(yaml_cfg_path, config_name='default', config_folder='cfg/')",
    "YamlConfig(yaml_cfg_path, config_name='default')"
)

with open('/content/CNDiff/scripts/run_cndiff.py', 'w') as f:
    f.write(content)

print("Done")

Done


In [78]:
%cd /content/CNDiff
!python3 -m scripts.run_cndiff --cfg ./cfg/exchange.yaml

/content/CNDiff
train 5202
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
val 745
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test 

In [79]:
!pip install yfinance

In [80]:
import yfinance as yf
import pandas as pd

# Nifty 50 stocks - multiple for multivariate
tickers = ['^NSEI', 'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS',
           'INFY.NS', 'ICICIBANK.NS', 'HINDUNILVR.NS', 'ITC.NS']

data = yf.download(tickers, start='2015-01-01', end='2024-12-31')['Close']
data = data.dropna()
print(data.shape)
print(data.head())
import pandas as pd
nse = data.reset_index()
nse.columns = ['date']+[str(i) for i in range(len(nse.columns)-1)]
cols = nse.columns.tolist()
cols[-1]= 'OT'
nse.columns = cols
nse['date'] = pd.to_datetime(nse['date']).dt.strftime('%Y/%m/%d 0:00')
print(nse.shape)
print(nse.head())
nse.to_csv('/content/CNDiff/data/nse_data.csv', index=False)
print("Saved")

/tmp/ipykernel_1727/2762892865.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start='2015-01-01', end='2024-12-31')['Close']
[*********************100%***********************]  8 of 8 completed


(2458, 8)
Ticker      HDFCBANK.NS  HINDUNILVR.NS  ICICIBANK.NS     INFY.NS      ITC.NS  \
Date                                                                           
2015-01-02   220.258667     631.084961    302.369690  374.998077  161.728821   
2015-01-05   218.399063     634.716370    302.995667  371.775543  162.365555   
2015-01-06   214.999176     646.737732    290.142975  364.008118  158.193863   
2015-01-07   215.626724     669.445129    282.297943  365.749817  155.251755   
2015-01-08   220.156006     682.092468    289.976166  367.593781  159.138000   

Ticker      RELIANCE.NS       TCS.NS        ^NSEI  
Date                                               
2015-01-02   189.496933  1002.049011  8395.450195  
2015-01-05   187.421265   986.820801  8378.400391  
2015-01-06   178.915253   950.440430  8127.350098  
2015-01-07   182.809814   939.213379  8102.100098  
2015-01-08   180.188492   949.352600  8234.599609  
(2458, 9)
              date           0           1           2 

In [81]:
with open('/content/CNDiff/cfg/exchange.yaml', 'r') as f:
    content = f.read()

content = content.replace("dataset: \"exchange\"", "dataset: \"nse\"")
content = content.replace("run_name: exchange", "run_name: nse")
content = content.replace("data_path: 'data/exchange_rate.csv'", "data_path: 'data/nse_data.csv'")
content = content.replace("feature_dim: 8", "feature_dim: 8")

with open('/content/CNDiff/cfg/nse.yaml', 'w') as f:
    f.write(content)

print("Done")


Done


In [82]:
%cd /content/CNDiff
!python3 -m scripts.run_cndiff --cfg ./cfg/nse.yaml

/content/CNDiff
train 1611
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
val 232
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test 

In [83]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/all_six_datasets/exchange_rate/exchange_rate.csv')
print(df.shape)
print(df.head())
print(df.columns.tolist())

(7588, 9)
            date       0       1         2         3         4         5  \
0  1990/1/1 0:00  0.7855  1.6110  0.861698  0.634196  0.211242  0.006838   
1  1990/1/2 0:00  0.7818  1.6100  0.861104  0.633513  0.211242  0.006863   
2  1990/1/3 0:00  0.7867  1.6293  0.861030  0.648508  0.211242  0.006975   
3  1990/1/4 0:00  0.7860  1.6370  0.862069  0.650618  0.211242  0.006953   
4  1990/1/5 0:00  0.7849  1.6530  0.861995  0.656254  0.211242  0.006940   

          6      OT  
0  0.525486  0.5930  
1  0.523972  0.5940  
2  0.526316  0.5973  
3  0.523834  0.5970  
4  0.527426  0.5985  
['date', '0', '1', '2', '3', '4', '5', '6', 'OT']


In [84]:
import pandas as pd
df = pd.read_csv('/content/CNDiff/data/nse_data.csv')
print(df.describe())


                 0            1            2            3            4  \
count  2458.000000  2458.000000  2458.000000  2458.000000  2458.000000   
mean    543.077700  1679.023550   533.294063   886.592956   223.152559   
std     197.899657   668.911502   312.269428   481.770927    92.201427   
min     214.999176   631.084961   155.355286   346.069733   107.234123   
25%     389.532463   962.066116   274.927170   431.962585   162.474354   
50%     537.823273  1842.326904   389.928635   635.941650   187.571144   
75%     720.565598  2285.292480   779.531952  1359.299866   240.894474   
max     923.393494  2939.454346  1335.792847  1942.221191   476.588745   

                 5            6            OT  
count  2458.000000  2458.000000   2458.000000  
mean    731.544260  2119.307315  13502.579601  
std     419.880351  1016.182444   4964.527352  
min     173.490646   844.116333   6970.600098  
25%     318.775528  1033.436035   9344.575195  
50%     642.846100  1829.863953  11513.800293

In [85]:
import numpy as np
df_log = df.copy()
for col in df.columns[1:]:  # skip date
    df_log[col] = np.log(df[col])
print(df_log.describe())

                 0            1            2            3            4  \
count  2458.000000  2458.000000  2458.000000  2458.000000  2458.000000   
mean      6.218066     7.327498     6.112955     6.632867     5.337266   
std       0.417610     0.469049     0.573753     0.560811     0.359040   
min       5.370634     6.447440     5.045715     5.846640     4.675015   
25%       5.964945     6.869080     5.616506     6.068339     5.090520   
50%       6.287530     7.518785     5.965964     6.455107     5.234158   
75%       6.580036     7.734249     6.658694     7.214725     5.484355   
max       6.828055     7.985979     7.197280     7.571588     6.166654   

                 5            6           OT  
count  2458.000000  2458.000000  2458.000000  
mean      6.387337     7.534580     9.446762  
std       0.687384     0.509010     0.353383  
min       5.156124     6.738290     8.849457  
25%       5.764384     6.940644     9.142551  
50%       6.465905     7.511997     9.351302  
75% 

In [86]:
df_log['date'] = df['date']  # date column wapas add karo
cols = ['date'] + [c for c in df_log.columns if c != 'date']
df_log = df_log[cols]
df_log.to_csv('/content/CNDiff/data/nse_log.csv', index=False)
print("Saved")

Saved


In [87]:
with open('/content/CNDiff/cfg/nse.yaml', 'r') as f:
    content = f.read()

content = content.replace("data_path: 'data/nse_data.csv'", "data_path: 'data/nse_log.csv'")
content = content.replace("run_name: nse", "run_name: nse_log")

with open('/content/CNDiff/cfg/nse_log.yaml', 'w') as f:
    f.write(content)
print("Done")

Done


In [88]:
%cd /content/CNDiff
!python3 -m scripts.run_cndiff --cfg ./cfg/nse.yaml

/content/CNDiff
train 1611
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
val 232
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test 

In [89]:
!python3 -m scripts.run_cndiff --cfg ./cfg/nse_log.yaml

train 1611
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
val 232
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
test 478
/usr/local/l

In [90]:
for seq_len in [48, 96, 192, 336]:
    print(f"\n--- seq_len: {seq_len} ---")
    !python3 -m scripts.run_cndiff --cfg ./cfg/nse_seq{seq_len}.yaml


--- seq_len: 48 ---
train 1659
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
val 232
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [91]:
!cat /content/CNDiff/train/train_non_gaussian.py

import numpy as np
import torch
import torch.nn as nn

import time
from tqdm import tqdm
from utils.diffusion_utils import extract

from models.model import EarlyStopping

###### preprocessing in dlinear
def instance_normalization(x, y0):
    x_mean = x[:,-1:,:] 
    x_std = torch.ones_like(x_mean)
    x_norm = (x - x_mean) / x_std
    y0_norm = (y0 - x_mean) / x_std
    return x_norm, y0_norm, x_mean, x_std

def instance_denormalization(y0, mean, std, pred_len):
    B = mean.shape[0]
    n_samples = y0.shape[0]//B
    std = torch.repeat_interleave(std, n_samples, dim=0).repeat(1, pred_len, 1)
    mean = torch.repeat_interleave(mean, n_samples, dim=0).repeat(1, pred_len, 1)
    y0 = y0 * std + mean
    return y0
    
class train(nn.Module):
    def __init__(self, config, train_dataloader, val_dataloader, test_dataloader, optimizer,scheduler, criterion, model):
        super(train, self).__init__()

        self.config = config
        self.optimizer = optimizer
        self.scheduler =

In [92]:
!sed -n '28,80p' /content/CNDiff/scripts/run_cndiff.py

config = pipe.read_conf()

# seed
random.seed(config.seed)
torch.manual_seed(config.seed)
np.random.seed(config.seed)

torch.backends.cudnn.deterministic = True
torch.cuda.manual_seed_all(config.seed)
torch.cuda.manual_seed(config.seed)
torch.backends.cudnn.benchmark =False

# Load Data
train_data, train_dataloader = load_data(config, flag='train')
val_data, val_dataloader = load_data(config, flag='val')
test_data, test_dataloader = load_data(config, flag='test')

# Loss function
criterion = nn.MSELoss()

# Load model
model = build_model(config)
optimizer = torch.optim.Adam([{'params': model.parameters()}], lr=config.train.lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min')

# train model
Trainer = train(config=config, train_dataloader=train_dataloader, val_dataloader=val_dataloader, test_dataloader=test_dataloader,
                 optimizer=optimizer,scheduler=scheduler, criterion=criterion, model=model)

# training
if config.test:
    final_model = Trai

In [93]:
with open('/content/CNDiff/scripts/run_cndiff.py', 'r') as f:
    content = f.read()

new_code = '''
# Save predictions for directional accuracy
import os
os.makedirs('predictions', exist_ok=True)
np.save(f'predictions/{config.run_name}_preds.npy', full_pred)
np.save(f'predictions/{config.run_name}_trues.npy', full_true)
print(f"Predictions saved: {full_pred.shape}, {full_true.shape}")
'''

content = content.replace(
    "print('mse:{}, mae:{}'.format(mse, mae))",
    "print('mse:{}, mae:{}'.format(mse, mae))" + new_code
)

# Better anchor point
content = content.replace(
    "f.close()",
    "f.close()" + new_code
)

with open('/content/CNDiff/scripts/run_cndiff.py', 'w') as f:
    f.write(content)

print("Done")

Done


In [94]:
for seq_len in [48, 96, 192, 336]:
    print(f"\n--- seq_len: {seq_len} ---")
    !python3 -m scripts.run_cndiff --cfg ./cfg/nse_seq{seq_len}.yaml


--- seq_len: 48 ---
train 1659
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
val 232
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [95]:
import numpy as np
import os
import re

# Ensure the current directory is CNDiff
%cd /content/CNDiff

# Define the sequence lengths
seq_lengths = [48, 96, 192, 336]

# Read the base config file
with open('cfg/nse_log.yaml', 'r') as f:
    base_config_content = f.read()

# Loop through each seq_len, create a new config, and run the model
for seq_len in seq_lengths:
    print(f"\n--- Preparing and running for seq_len: {seq_len} ---")

    # Replace seq_len and run_name in the base config
    current_config_content = re.sub(r"seq_len: \d+", f"seq_len: {seq_len}", base_config_content)
    current_config_content = re.sub(r"run_name: nse_log", f"run_name: nse_seq{seq_len}", current_config_content)

    # Define the new config file path
    new_config_path = f"./cfg/nse_seq{seq_len}.yaml"

    # Write the new config to a file
    with open(new_config_path, 'w') as f:
        f.write(current_config_content)
    print(f"Created config: {new_config_path}")

    # Run the model with the new config
    !python3 -m scripts.run_cndiff --cfg {new_config_path}

# After all runs, attempt to load the predictions for nse_seq96
try:
    preds = np.load('/content/CNDiff/predictions/nse_seq96_preds.npy')
    trues = np.load('/content/CNDiff/predictions/nse_seq96_trues.npy')

    print("\nPreds shape:", preds.shape)
    print("Trues shape:", trues.shape)

    # Optional: Display a snippet of the data
    print("\nPredictions head:\n", preds[:5])
    print("\nTrue values head:\n", trues[:5])
except FileNotFoundError:
    print(f"Error: Predictions for nse_seq96 still not found after running all configurations.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

/content/CNDiff

--- Preparing and running for seq_len: 48 ---
Created config: ./cfg/nse_seq48.yaml
train 1659
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
val 232
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid po

In [96]:
import numpy as np

preds = np.load('/content/CNDiff/predictions/nse_seq96_preds.npy')
trues = np.load('/content/CNDiff/predictions/nse_seq96_trues.npy')

# preds shape: (14, 32, 10, 14, 8) — 10 copies
# trues shape: (14, 32, 14, 8)

# Mean over copies (dim 2)
preds_mean = preds.mean(axis=2)  # (14, 32, 14, 8)

# Flatten batch dimensions
preds_flat = preds_mean.reshape(-1, 14, 8)  # (N, pred_len, features)
trues_flat = trues.reshape(-1, 14, 8)

# Directional accuracy — did prediction go same direction as actual?
pred_direction = np.sign(preds_flat[:, 1:, :] - preds_flat[:, :-1, :])
true_direction = np.sign(trues_flat[:, 1:, :] - trues_flat[:, :-1, :])

directional_acc = (pred_direction == true_direction).mean()
print(f"Directional Accuracy: {directional_acc:.4f} ({directional_acc*100:.2f}%)")

# Per feature
for i in range(8):
    acc = (pred_direction[:,:,i] == true_direction[:,:,i]).mean()
    print(f"Feature {i}: {acc*100:.2f}%")



Directional Accuracy: 0.5020 (50.20%)
Feature 0: 50.84%
Feature 1: 49.73%
Feature 2: 50.62%
Feature 3: 49.95%
Feature 4: 49.16%
Feature 5: 50.46%
Feature 6: 50.36%
Feature 7: 50.50%


In [97]:
"""judging from. the directional accuracy of the features, one clear observation that can be seen here is that
despite the low mae and mse, we can't really predict whether the stock value goes up or down,
coz 50% accuracy isnt any differnet from being random here
so now our bigger objective is to fix dimentional accuracy
"""

"judging from. the directional accuracy of the features, one clear observation that can be seen here is that\ndespite the low mae and mse, we can't really predict whether the stock value goes up or down, \ncoz 50% accuracy isnt any differnet from being random here\nso now our bigger objective is to fix dimentional accuracy\n"